# Датасет для завдання

In [17]:
import sys
import os
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

data_csv = str(Path().resolve().parent / "data" / "nuek-vuh3.csv")
print(data_csv)

/Users/kozak_mamay/PycharmProjects/WoolfDataScience/home_works/data_engineering/data/nuek-vuh3.csv


# Частина 1

In [24]:
from pyspark.sql import SparkSession

# Створюємо сесію Spark
spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .appName("MyGoitSparkSandbox") \
    .getOrCreate()

# Завантажуємо датасет
nuek_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(data_csv)

nuek_repart = nuek_df.repartition(2)

nuek_processed = nuek_repart \
    .where("final_priority < 3") \
    .select("unit_id", "final_priority") \
    .groupBy("unit_id") \
    .count()

# Ось ТУТ додано рядок
nuek_processed = nuek_processed.where("count>2")

nuek_processed.collect()

input("Press Enter to continue...5")

# Закриваємо сесію Spark
spark.stop()


В цьому випадку ми маємо 5 jobs. Ось їх послідовність
1. Завантаження датасету (перший прохід по файлу)
2. Завантаження датасету (другий прохід по файлу)
3. Читання датасету + фільтрація
4. Шуфл від репатриювання
5. Шуфл від where

# Частина 2

In [20]:
from pyspark.sql import SparkSession

# Створюємо сесію Spark
spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .appName("MyGoitSparkSandbox") \
    .getOrCreate()

# Завантажуємо датасет
nuek_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(data_csv)

nuek_repart = nuek_df.repartition(2)

nuek_processed = nuek_repart \
    .where("final_priority < 3") \
    .select("unit_id", "final_priority") \
    .groupBy("unit_id") \
    .count()

# Проміжний action: collect
nuek_processed.collect()

# Ось ТУТ додано рядок
nuek_processed = nuek_processed.where("count>2")

nuek_processed.collect()

input("Press Enter to continue...5")

# Закриваємо сесію Spark
spark.stop()


У цьому випадку ми маємо 8 jobs:
1. Завантаження датасету (перший прохід по файлу)
2. Завантаження датасету (другий прохід по файлу)
3. Repartition shuffle
4. Читання даних + select
5. count
6. Читання даних + collect
7. Фільтрація результатів
8. collect

# Частина 3

In [23]:
from pyspark.sql import SparkSession

# Створюємо сесію Spark
spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "2") \
    .appName("MyGoitSparkSandbox") \
    .getOrCreate()

# Завантажуємо датасет
nuek_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(data_csv)

nuek_repart = nuek_df.repartition(2)

nuek_processed_cached = nuek_repart \
    .where("final_priority < 3") \
    .select("unit_id", "final_priority") \
    .groupBy("unit_id") \
    .count() \
    .cache()  # Додано функцію cache

# Проміжний action: collect
nuek_processed_cached.collect()

# Ось ТУТ додано рядок
nuek_processed = nuek_processed_cached.where("count>2")

nuek_processed.collect()

input("Press Enter to continue...5")

# Звільняємо пям'ять від Dataframe
nuek_processed_cached.unpersist()

# Закриваємо сесію Spark
spark.stop()


Тут маємо 7 jobs:
1. Завантаження датасету (перший прохід по файлу)
2. Завантаження датасету (другий прохід по файлу)
3. Repartition shuffle
4. Читання даних + фльтрація
5. select
6. count
7. collect